# Long-Read Assembly & Structural Variants
**Tier 3 — Applied Bioinformatics | Module 25 · Notebook 2**

*Prerequisites: Notebook 1 (ONT Processing), Module 17 (Genome Assembly)*

---

**By the end of this notebook you will be able to:**

1. Assemble a genome or metagenome from ONT reads with Flye
2. Assemble HiFi data with Hifiasm and understand the difference from ONT assembly
3. Polish an ONT assembly with Medaka (neural-network polishing)
4. Assess assembly quality with QUAST and BUSCO
5. Detect structural variants (deletions, insertions, inversions) with Sniffles2

**Key resources:**
- [Flye documentation](https://github.com/fenderglass/Flye)
- [Hifiasm documentation](https://github.com/chhylp123/hifiasm)
- [Sniffles2 documentation](https://github.com/fritzsedlazeck/Sniffles)

---

[← ONT Processing](./01_ont_processing.ipynb) | [Module Overview](../README.md) | [Next: Isoform Analysis →](./03_isoform_analysis.ipynb)

## 1. De Novo Assembly with Flye

Flye uses a **repeat graph** assembly algorithm. Unlike overlap-layout-consensus (OLC) assemblers, Flye constructs an assembly graph where repeated sequences are collapsed into shared nodes. This handles the complex repeat structure of eukaryotic genomes better than traditional OLC approaches, because repeats that would cause an OLC graph to collapse into a tangled hairball are instead represented as traversed-multiple-times edges. The underlying graph is iterative: Flye first builds an approximate repeat graph from a disjointig set, then resolves it by checking which paths through repeat nodes are supported by bridging reads.

Input modes control how Flye models the error profile of your reads. Use `--nano-hq` for current ONT R10.4.1 reads (Q20+), `--nano-raw` for older R9.4.1 data (Q < 15), `--nano-corr` for ONT reads that have already been error-corrected by another tool, `--pacbio-hifi` for PacBio CCS/HiFi reads (Q20+), and `--pacbio-raw` for older PacBio CLR reads. Choosing the wrong mode significantly degrades assembly quality because Flye tunes its overlap finding and error tolerance per mode.

The `--genome-size` flag is required (e.g., `5m` for 5 Mb bacterial, `3g` for 3 Gb human). Flye uses this estimate to set expected coverage depth and to calibrate repeat detection thresholds. It does not need to be exact — values within ±50% are generally fine. The Flye output directory contains: `assembly.fasta` (final contigs), `assembly_graph.gfa` (assembly graph for visualization in Bandage), and `assembly_info.txt` (per-contig length, coverage, circularity, and multiplicity statistics).

| Organism | Input | Expected contigs | Typical N50 |
|---|---|---|---|
| Bacterial (5 Mb) | ONT R10.4.1, 100× | 1–5 contigs | ~1–5 Mb (near-complete) |
| Yeast (12 Mb) | ONT R10.4.1, 50× | 20–50 contigs | ~500 kb – 1 Mb |
| Human (3 Gb) | ONT R10.4.1, 30× | 1,000–3,000 contigs | ~40–80 Mb |
| Human (3 Gb) | PacBio HiFi, 30× | 500–1,500 contigs | ~80–150 Mb |
| Metagenome | ONT R10.4.1, mixed | depends on diversity | 100 kb – 5 Mb (per species) |

In [ ]:
# Assemble ONT R10.4.1 reads (high quality mode)
# !flye --nano-hq calls_filtered.fastq.gz \
#       --genome-size 5m \
#       --out-dir flye_assembly/ \
#       --threads 8

# For lower quality ONT reads (R9.4.1, Q < 15):
# !flye --nano-raw calls_filtered.fastq.gz --genome-size 5m --out-dir flye_r9/ --threads 8

# For PacBio HiFi reads:
# !flye --pacbio-hifi hifi_reads.fastq.gz --genome-size 5m --out-dir flye_hifi/ --threads 8

# View assembly statistics
# !cat flye_assembly/assembly_info.txt
# !grep -c ">" flye_assembly/assembly.fasta  # count contigs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Simulate a bacterial assembly result (5 Mb genome, ONT R10.4.1)
# Flye typically produces near-complete assemblies for bacteria
n_contigs = rng.poisson(3)  # typically 1-5 contigs
n_contigs = max(n_contigs, 1)

genome_size = 5_000_000
# Simulate contig lengths that sum to approximately genome size
raw_lengths = rng.exponential(genome_size / n_contigs, n_contigs)
contig_lengths = (raw_lengths / raw_lengths.sum() * genome_size).astype(int)
# Fix rounding
contig_lengths[-1] = genome_size - contig_lengths[:-1].sum()
contig_lengths = np.sort(contig_lengths)[::-1]

# Generate coverage (depth ~ 100x)
mean_cov = 100.0
coverages = rng.normal(mean_cov, mean_cov * 0.05, n_contigs)

# Build assembly_info.txt style table
df_info = pd.DataFrame({
    "seq_name": [f"contig_{i+1:03d}" for i in range(n_contigs)],
    "length": contig_lengths,
    "cov.": coverages.round(1),
    "circ.": ["Y" if l > 1_000_000 else "N" for l in contig_lengths],
    "repeat": ["N"] * n_contigs,
    "mult.": [1] * n_contigs,
    "alt_group": ["*"] * n_contigs,
    "graph_path": [f"edge_{i+1}" for i in range(n_contigs)],
})

print("=== Flye assembly_info.txt (simulated) ===")
print(df_info.to_string(index=False))
print(f"\nTotal assembly size: {contig_lengths.sum():,} bp")
print(f"Number of contigs: {n_contigs}")
print(f"Largest contig:    {contig_lengths[0]:,} bp ({contig_lengths[0]/genome_size*100:.1f}% of genome)")
print(f"Mean coverage:     {coverages.mean():.1f}\u00d7")

# N50 calculation
def compute_n50(lengths):
    sorted_l = np.sort(lengths)[::-1]
    cumsum = np.cumsum(sorted_l)
    idx = np.searchsorted(cumsum, cumsum[-1] / 2)
    return sorted_l[idx]

n50 = compute_n50(contig_lengths)
print(f"Assembly N50:      {n50:,} bp")

## 2. HiFi Assembly with Hifiasm

Hifiasm is the leading assembler for PacBio HiFi reads and can also assemble Hi-C+HiFi or HiFi+ONT ultra-long combinations. It uses a **haplotype-resolved** assembly graph that produces **phased** assemblies by default. Rather than collapsing heterozygous loci into a consensus, Hifiasm preserves both haplotypes as separate paths through the graph, enabling downstream allele-specific analyses that are impossible with collapsed assemblers.

HiFi reads are ~15–20 kb with Q20+ accuracy (>99%). This combination allows Hifiasm to construct assembly graphs with far fewer gaps than ONT-only assemblies. The output format is **GFA (Graph Fragment Assembly)** files, not FASTA directly — the primary assembly contigs are in `*.bp.p_ctg.gfa`. Convert to FASTA with a simple awk one-liner: `awk '/^S/{print ">"$2"\n"$3}'`. The key output files are: `sample.asm.bp.p_ctg.gfa` (primary contigs, one representative haplotype), `sample.asm.bp.hap1.p_ctg.gfa` (haplotype 1), and `sample.asm.bp.hap2.p_ctg.gfa` (haplotype 2).

**Trio binning** is available when parental Illumina reads are provided. Hifiasm uses yak to build k-mer frequency databases from each parent, then assigns each HiFi read to a parental haplotype before assembly. This produces two fully phased haplotype assemblies (paternal and maternal) — critical for allele-specific gene expression, imprinting analysis, and compound heterozygous variant phasing in clinical genetics.

Compared to Flye, Hifiasm typically produces larger contigs (higher N50) and better phasing from HiFi data. Flye remains more versatile — it accepts many read types including ONT, CLR, and mixed inputs — and is the preferred choice for metagenomics and samples where only ONT data is available. For human genomes with HiFi data, Hifiasm is generally the first choice.

| Feature | Flye | Hifiasm |
|---|---|---|
| Read types | ONT, HiFi, CLR, mixed | HiFi primary; HiFi+ONT UL |
| Phasing | Collapsed (one haplotype) | Haplotype-resolved by default |
| Metagenomics | Yes (`--meta` flag) | No |
| Typical human N50 (HiFi 30×) | ~80–120 Mb | ~120–180 Mb |
| Trio binning | No | Yes (with yak) |
| Hi-C scaffolding | Via 3rd-party | Native support |

In [ ]:
# Standard HiFi assembly with Hifiasm
# !hifiasm -o sample.asm -t 8 hifi_reads.fastq.gz

# Convert primary assembly GFA to FASTA
# !awk '/^S/{print ">"$2"\n"$3}' sample.asm.bp.p_ctg.gfa > assembly_primary.fasta

# Trio-binned phased assembly (requires parental k-mer databases)
# !yak count -b37 -t 8 -o pat.yak paternal_illumina.fastq.gz
# !yak count -b37 -t 8 -o mat.yak maternal_illumina.fastq.gz
# !hifiasm -o sample_trio.asm -t 8 -1 pat.yak -2 mat.yak hifi_reads.fastq.gz
# !awk '/^S/{print ">"$2"\n"$3}' sample_trio.asm.dip.hap1.p_ctg.gfa > hap1.fasta
# !awk '/^S/{print ">"$2"\n"$3}' sample_trio.asm.dip.hap2.p_ctg.gfa > hap2.fasta

# Count and summarize contigs
# !grep -c ">" assembly_primary.fasta
# !awk '/^>/{next}{len+=length($0)} END{print "Total bases:", len}' assembly_primary.fasta

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(10)

# Simulate contig length distributions for different assemblers (human genome, chr1-scale)
def sim_contigs(n_contigs, mean_len, cv=0.8, genome_size=3e9):
    """Simulate contig lengths."""
    lengths = rng.lognormal(np.log(mean_len), cv, n_contigs)
    lengths = (lengths / lengths.sum() * genome_size).astype(int)
    return np.sort(lengths)[::-1]

assemblers = {
    "Flye (ONT R10.4.1)":        sim_contigs(1200, 2_000_000, 1.0),
    "Hifiasm (PacBio HiFi)":     sim_contigs(400,  6_000_000, 0.9),
    "Hifiasm (HiFi + ONT UL)":   sim_contigs(150, 15_000_000, 0.8),
}

def n50(lengths):
    s = np.sort(lengths)[::-1]
    cs = np.cumsum(s)
    return s[np.searchsorted(cs, cs[-1]/2)]

print(f"{'Assembler':<28} {'Contigs':>8} {'Total (Gb)':>10} {'N50 (Mb)':>10} {'Largest (Mb)':>13}")
print("-" * 72)
for name, lens in assemblers.items():
    print(f"{name:<28} {len(lens):>8,} {lens.sum()/1e9:>10.2f} {n50(lens)/1e6:>10.1f} {lens[0]/1e6:>13.1f}")

# Plot NG50 plots (cumulative contig length vs sorted contig length)
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["steelblue", "darkorange", "seagreen"]
for (name, lens), color in zip(assemblers.items(), colors):
    sorted_l = np.sort(lens)[::-1]
    cumulative = np.cumsum(sorted_l) / 3e9 * 100  # % of genome size
    ax.plot(cumulative, sorted_l / 1e6, label=name, color=color, linewidth=2)

ax.axhline(1, color="gray", linestyle=":", alpha=0.5, label="1 Mb reference line")
ax.set_xlabel("Cumulative genome coverage (%)")
ax.set_ylabel("Contig length (Mb)")
ax.set_title("Assembly contig length distributions (simulated, human genome scale)")
ax.legend()
ax.set_xlim(0, 100)
ax.set_ylim(0)
plt.tight_layout()
plt.show()

## 3. Assembly Polishing with Medaka

Polishing corrects remaining systematic errors in the draft assembly. For ONT assemblies, Flye's draft typically has ~0.1–1% error rate, dominated by homopolymer indels — insertions or deletions within runs of the same base (e.g., AAAAAAA → AAAAAA). These are characteristic of nanopore sequencing because the electrical signal changes only when a new base enters the sensing region, making homopolymer runs inherently ambiguous. Short-read polishing tools like Pilon can also address these errors if Illumina data is available.

Medaka is ONT's neural-network polisher. It maps the original reads back to the draft assembly using Minimap2, then uses a recurrent neural network (RNN) — specifically a gated recurrent unit (GRU) architecture — to call consensus corrections from the per-position pileup. Critically, Medaka requires that the model matches the flow cell chemistry used during sequencing. The model naming convention encodes chemistry: `r1041_e82_400bps_hac_v4.2.0` = pore R10.4.1, chemistry E8.2, 400 bps translocation speed, HAC basecalling model, version 4.2.0. Using the wrong model degrades polishing performance because the error signatures differ between chemistries.

Medaka is specifically designed for ONT reads. For PacBio HiFi assemblies, polishing is usually unnecessary because HiFi reads are already Q20+ and the assembly consensus is already very accurate. For ONT assemblies with <30× coverage, Medaka may not help much — insufficient depth means the RNN cannot distinguish true variants from sequencing errors. One round of polishing is usually sufficient for high-depth (>50×) ONT data; additional rounds give diminishing returns and can introduce systematic biases.

| Error type | Flye draft | After Medaka | Improvement |
|---|---|---|---|
| Substitutions | ~0.03% | ~0.005% | ~6× |
| Insertions | ~0.15% | ~0.02% | ~7× |
| Deletions | ~0.10% | ~0.015% | ~7× |
| Homopolymer errors | ~0.5–1% | ~0.1–0.2% | ~5× |
| Overall QV | ~QV30 | ~QV40–45 | significant |

In [ ]:
# Medaka polishing — must match your flow cell chemistry/model
# !medaka_consensus \
#     -i calls_filtered.fastq.gz \   # original reads
#     -d flye_assembly/assembly.fasta \  # draft assembly
#     -o medaka_polished/ \
#     -t 8 \
#     -m r1041_e82_400bps_hac_v4.2.0   # IMPORTANT: must match flow cell

# List available Medaka models (check which matches your run)
# !medaka tools list_models

# Output: medaka_polished/consensus.fasta

# Quick QV estimate using k-mer completeness (Merqury)
# !merqury.sh hifi_reads.fastq.gz medaka_polished/consensus.fasta merqury_out

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Simulated QUAST comparison: Flye draft vs Medaka polished (bacterial genome)
data = {
    "Metric": [
        "Genome fraction (%)",
        "Largest contig (bp)",
        "N50 (bp)",
        "# contigs (>= 500 bp)",
        "# misassemblies",
        "# mismatches per 100 kbp",
        "# indels per 100 kbp",
        "Duplication ratio",
        "GC (%)",
    ],
    "Flye draft": [
        98.4, 4_812_033, 4_812_033, 3, 1, 48.2, 162.7, 1.002, 50.7
    ],
    "Medaka polished": [
        99.1, 4_822_105, 4_822_105, 2, 0, 5.4, 14.1, 1.001, 50.8
    ],
    "Reference": [
        100, 5_000_000, 5_000_000, 1, 0, 0, 0, 1.000, 50.7
    ]
}
df_quast = pd.DataFrame(data)

print("=== QUAST comparison: Flye draft vs Medaka polished ===\n")
print(df_quast.to_string(index=False))

# Plot error rate improvement
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
error_metrics = ["# mismatches per 100 kbp", "# indels per 100 kbp"]
for ax, metric in zip(axes, error_metrics):
    vals = [
        df_quast.loc[df_quast["Metric"]==metric, "Flye draft"].values[0],
        df_quast.loc[df_quast["Metric"]==metric, "Medaka polished"].values[0],
    ]
    bars = ax.bar(["Flye draft", "Medaka polished"], vals,
                  color=["steelblue", "seagreen"], edgecolor="white")
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{val:.1f}", ha="center", va="bottom", fontsize=11)
    ax.set_title(metric)
    ax.set_ylabel("Errors per 100 kbp")
    ax.set_ylim(0, max(vals) * 1.3)

plt.suptitle("Assembly error rates before and after Medaka polishing (simulated)", fontweight="bold")
plt.tight_layout()
plt.show()

mismatch_draft = df_quast.loc[df_quast["Metric"]=="# mismatches per 100 kbp", "Flye draft"].values[0]
mismatch_pol   = df_quast.loc[df_quast["Metric"]=="# mismatches per 100 kbp", "Medaka polished"].values[0]
print(f"\nMismatch reduction: {mismatch_draft/mismatch_pol:.1f}\u00d7 improvement after Medaka polishing")

## 4. Assembly Quality Assessment

QUAST (Quality Assessment Tool for Genome Assemblies) compares an assembly against a reference genome (if available) and computes a rich set of contiguity and accuracy metrics. **N50** is the contig length at which 50% of all assembled bases are contained in contigs at least this long — higher values indicate better contiguity. **NG50** is N50 normalized to the known or expected genome size, making it more meaningful when the assembly may be incomplete. **Genome fraction** reports what percentage of the reference is covered by the assembly. **# misassemblies** counts contigs that contradict the reference due to inversions, translocations, or large-scale relocations. **# mismatches and indels per 100 kbp** estimate the per-base error rate relative to the reference.

BUSCO (Benchmarking Universal Single-Copy Orthologs) measures gene-space completeness independently of a reference genome. It searches the assembly for lineage-specific conserved single-copy genes from the OrthoDB database and classifies each as: **Complete (C)** — found once at expected length; **Duplicated (D)** — found more than once (may indicate assembly redundancy or true gene duplication); **Fragmented (F)** — found but truncated; **Missing (M)** — not found. A good bacterial assembly should have C > 95%, D < 2%. A good eukaryotic genome should have C > 90%, D < 5%. Choose the lineage database appropriate for your organism: `bacteria_odb10`, `fungi_odb10`, `eukaryota_odb10`, `mammalia_odb10`, `vertebrata_odb10`, etc.

QUAST and BUSCO are complementary: QUAST requires a reference and measures structural accuracy and contiguity; BUSCO is reference-free and measures biological completeness at the gene level. Together they provide a comprehensive quality picture. For assemblies without a reference, BUSCO is especially valuable. For assemblies of well-characterized model organisms, QUAST provides the most detailed breakdown of assembly errors.

| Tool | Reference needed | Measures | Key outputs |
|---|---|---|---|
| QUAST | Optional (better with) | Contiguity, structural accuracy | N50, NG50, misassemblies, mismatches/indels |
| BUSCO | No | Gene-space completeness | C/D/F/M percentages |
| Merqury | Short reads (k-mers) | Base-level accuracy (QV) | QV score, k-mer completeness |
| Inspector | Long reads | Local error rate | Error profile per contig |

In [ ]:
# QUAST — compare draft vs polished, with reference genome
# !quast.py flye_assembly/assembly.fasta medaka_polished/consensus.fasta \
#     --reference reference.fasta \
#     --output-dir quast_report/ \
#     --threads 8

# View HTML report (in a local environment)
# !open quast_report/report.html

# BUSCO completeness assessment
# !busco -i medaka_polished/consensus.fasta \
#        -l bacteria_odb10 \
#        -o busco_out/ \
#        -m genome \
#        --cpu 8

# For eukaryotes:
# !busco -i assembly.fasta -l eukaryota_odb10 -o busco_euk/ -m genome --cpu 8

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Utility: compute N-statistics from an array of contig lengths
# ------------------------------------------------------------------
def n_statistic(lengths, genome_size=None, thresholds=(10, 25, 50, 75, 90)):
    """
    Compute N-statistics (N10, N25, N50, N75, N90) for an assembly.
    If genome_size is provided, also compute NG-statistics.
    """
    sorted_l = np.sort(np.array(lengths, dtype=int))[::-1]
    total = sorted_l.sum()
    reference = genome_size if genome_size else total
    cumsum = np.cumsum(sorted_l)
    results = {}
    for t in thresholds:
        target = reference * t / 100
        idx = np.searchsorted(cumsum, target)
        idx = min(idx, len(sorted_l) - 1)
        results[f"N{t}"] = sorted_l[idx]
        if genome_size:
            results[f"NG{t}"] = sorted_l[idx]
    results["L50"] = int(np.searchsorted(cumsum, total * 0.5)) + 1
    results["# contigs"] = len(sorted_l)
    results["Total (Mb)"] = total / 1e6
    results["Largest (Mb)"] = sorted_l[0] / 1e6
    return results

rng = np.random.default_rng(7)

# Simulate three assemblies
assemblies = {
    "Flye draft (ONT)":      rng.lognormal(13.0, 1.2, 40).astype(int),
    "Medaka polished":       rng.lognormal(13.2, 1.1, 32).astype(int),
    "Hifiasm (HiFi)":        rng.lognormal(14.5, 0.9, 10).astype(int),
}

genome_size = 50_000_000  # 50 Mb example genome
print(f"{'Assembly':<28} {'N50 (Mb)':>9} {'NG50 (Mb)':>10} {'# ctg':>6} {'Total (Mb)':>11} {'Largest (Mb)':>13}")
print("-" * 82)
for name, lens in assemblies.items():
    s = n_statistic(lens, genome_size=genome_size)
    print(f"{name:<28} {s['N50']/1e6:>9.2f} {s['NG50']/1e6:>10.2f} {s['# contigs']:>6} {s['Total (Mb)']:>11.1f} {s['Largest (Mb)']:>13.2f}")

# BUSCO-style plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# N50 comparison bar chart
n50_vals = {name: n_statistic(lens, genome_size)["N50"]/1e6 for name, lens in assemblies.items()}
axes[0].barh(list(n50_vals.keys()), list(n50_vals.values()),
             color=["steelblue", "seagreen", "darkorange"])
axes[0].set_xlabel("N50 (Mb)")
axes[0].set_title("Assembly N50 comparison")
axes[0].axvline(1, color="gray", linestyle=":", alpha=0.5)

# Simulated BUSCO results
busco_data = {
    "Flye draft":       {"Complete": 91.2, "Duplicated": 1.8, "Fragmented": 4.1, "Missing": 2.9},
    "Medaka polished":  {"Complete": 96.4, "Duplicated": 1.5, "Fragmented": 1.8, "Missing": 0.3},
    "Hifiasm (HiFi)":   {"Complete": 98.1, "Duplicated": 1.2, "Fragmented": 0.5, "Missing": 0.2},
}
categories = ["Complete", "Duplicated", "Fragmented", "Missing"]
colors_busco = ["#2ecc71", "#f39c12", "#e74c3c", "#95a5a6"]
y_pos = np.arange(len(busco_data))
bottoms = np.zeros(len(busco_data))
for cat, col in zip(categories, colors_busco):
    vals_b = [busco_data[name][cat] for name in busco_data]
    axes[1].barh(y_pos, vals_b, left=bottoms, color=col, label=cat)
    bottoms += np.array(vals_b)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(list(busco_data.keys()))
axes[1].set_xlabel("BUSCO genes (%)")
axes[1].set_title("BUSCO completeness (bacteria_odb10, simulated)")
axes[1].legend(loc="lower right", fontsize=9)
axes[1].set_xlim(0, 101)

plt.suptitle("Assembly quality assessment (simulated)", fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Structural Variant Calling with Sniffles2

Structural variants (SVs) are genomic rearrangements ≥50 bp. Short-read sequencing detects SVs indirectly through discordant read pairs and split reads, which limits sensitivity — particularly for SVs in repetitive regions where short reads mis-map. Long reads directly span most SVs, making long-read SV calling far more sensitive and precise. Sniffles2 is currently the leading long-read SV caller, supporting both single-sample and population-scale multi-sample analysis.

Sniffles2 detects five major SV classes. **DEL** (deletions) are the most common and most reliably detected. **INS** (insertions) include mobile element insertions (MEIs) such as Alu (~282 bp) and L1 (~6 kb) retrotransposons that are pervasive in the human genome. **INV** (inversions) flip a segment's orientation and are important in cancer and complex structural rearrangements. **DUP** (tandem or interspersed duplications) increase copy number. **BND** (breakends) represent complex rearrangements and translocations that do not fit the other categories.

The Sniffles2 VCF output uses standard SV INFO fields. `SVTYPE` gives the class; `SVLEN` gives the size (negative for deletions); `SUPPORT` gives the number of supporting reads; `AF` gives the allele frequency (0.5 = heterozygous, 1.0 = homozygous alt); `STRAND` reports strand bias. Filtering on `SUPPORT >= 5` and `FILTER = PASS` is a standard starting point for high-confidence calls.

A key Sniffles2 feature for population studies is the **SNF (Sniffles Format)** file. Each sample generates one `.snf` file during single-sample calling, and any number of `.snf` files can be jointly genotyped in a single multi-sample Sniffles2 run without re-running alignment. This is far more efficient than VCF merging with tools like SURVIVOR for large cohorts. A typical human genome (30× ONT) yields approximately 15,000 DEL, 17,000 INS (many mobile elements), 500 DUP, and 200 INV.

| SV type | Typical count (human) | Typical size range | Detection sensitivity (ONT 30×) |
|---|---|---|---|
| DEL | ~15,000 | 50 bp – 1 Mb | >90% for >100 bp |
| INS | ~17,000 | 50 bp – 10 kb | >85% for >100 bp |
| DUP | ~500 | 1 kb – 1 Mb | ~70–80% |
| INV | ~200 | 1 kb – 10 Mb | ~80% |
| BND | ~180 | — | ~60–75% |

In [ ]:
# Single-sample SV calling
# !sniffles --input ont_aligned.bam \
#           --vcf svs.vcf \
#           --reference hg38.fa \
#           --threads 8 \
#           --snf sample.snf      # save SNF for multi-sample joint calling

# Multi-sample joint genotyping (faster than re-running per sample)
# !sniffles --input sample1.snf sample2.snf sample3.snf \
#           --vcf joint_svs.vcf \
#           --reference hg38.fa

# Filter for high-confidence calls
# !bcftools view -i 'FILTER="PASS" && INFO/SUPPORT>=5' svs.vcf > svs_filtered.vcf

# Count SV types
# !bcftools query -f '%INFO/SVTYPE\n' svs_filtered.vcf | sort | uniq -c | sort -rn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

rng = np.random.default_rng(3)

# Simulate a realistic SV callset (human genome, ONT 30x coverage)
sv_counts = {
    "DEL": 14_800,
    "INS": 16_500,
    "DUP": 520,
    "INV": 210,
    "BND": 180,
}

sv_dfs = []
for svtype, n in sv_counts.items():
    if svtype == "DEL":
        sizes = rng.lognormal(6.5, 1.4, n).astype(int)  # median ~600 bp
        afs   = rng.beta(1.5, 2.5, n)
    elif svtype == "INS":
        # Many mobile element insertions (L1, Alu) -> bimodal
        sizes_alu   = rng.normal(282, 20, int(n * 0.35)).astype(int)    # Alu ~282 bp
        sizes_l1    = rng.normal(6000, 1500, int(n * 0.20)).astype(int) # L1HS ~6 kb
        sizes_other = rng.lognormal(5.5, 1.2, n - len(sizes_alu) - len(sizes_l1)).astype(int)
        sizes = np.concatenate([sizes_alu, sizes_l1, sizes_other])
        afs = rng.beta(1.2, 2.2, len(sizes))
        n = len(sizes)
    elif svtype == "DUP":
        sizes = rng.lognormal(8.5, 1.5, n).astype(int)
        afs = rng.beta(1, 3, n)
    elif svtype == "INV":
        sizes = rng.lognormal(9.0, 1.5, n).astype(int)
        afs = rng.beta(0.8, 2, n)
    else:  # BND
        sizes = rng.integers(50, 500, n)
        afs = rng.beta(0.5, 2, n)
    support = rng.poisson(20, n)
    sv_dfs.append(pd.DataFrame({
        "SVTYPE": svtype,
        "SVLEN":    np.abs(sizes[:n]),
        "AF":       np.clip(afs[:n], 0.01, 1.0),
        "SUPPORT":  np.clip(support, 3, 100),
    }))

df_sv = pd.concat(sv_dfs, ignore_index=True)

print(f"Total SVs called: {len(df_sv):,}")
print("\nSV type distribution:")
for svtype, count in sorted(sv_counts.items(), key=lambda x: -x[1]):
    print(f"  {svtype:<6} {count:>7,}  ({count/len(df_sv)*100:.1f}%)")

# Figure
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: SV type counts
types   = list(sv_counts.keys())
counts  = list(sv_counts.values())
colors  = ["#e74c3c","#3498db","#f39c12","#2ecc71","#9b59b6"]
axes[0].bar(types, counts, color=colors, edgecolor="white")
axes[0].set_ylabel("Number of SVs")
axes[0].set_title("SV type distribution")
for i, (t, c) in enumerate(zip(types, counts)):
    axes[0].text(i, c + 100, f"{c:,}", ha="center", va="bottom", fontsize=9)

# Panel 2: SV size distribution (DEL and INS)
for svtype, color in [("DEL","#e74c3c"), ("INS","#3498db")]:
    sizes = df_sv.loc[df_sv.SVTYPE == svtype, "SVLEN"]
    axes[1].hist(np.log10(sizes.clip(50, 1e7)), bins=60,
                 alpha=0.6, color=color, label=svtype, density=True)
axes[1].set_xlabel("SV size (log\u2081\u2080 bp)")
axes[1].set_ylabel("Density")
axes[1].set_title("DEL and INS size distribution")
axes[1].set_xticks([2, 3, 4, 5, 6])
axes[1].set_xticklabels(["100 bp","1 kb","10 kb","100 kb","1 Mb"])
axes[1].legend()

# Panel 3: Allele frequency spectrum
axes[2].hist(df_sv["AF"], bins=40, color="steelblue", edgecolor="white", lw=0.3)
axes[2].axvline(0.5, color="crimson", linestyle="--", label="Het (AF=0.5)")
axes[2].set_xlabel("Allele frequency")
axes[2].set_ylabel("Count")
axes[2].set_title("SV allele frequency spectrum")
axes[2].legend()

plt.suptitle("Sniffles2 SV callset (simulated, human genome 30\u00d7 ONT)", fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

Long-read sequencing enables high-quality de novo genome assembly with Flye (ONT) or Hifiasm (HiFi), far outperforming short-read-only approaches for repetitive regions, centromeres, and complex structural variants. The choice of assembler depends primarily on read type: Flye is versatile and handles ONT, CLR, and metagenomics; Hifiasm produces superior phased assemblies from HiFi data, especially when combined with Hi-C or ultra-long ONT reads.

Medaka polishing significantly reduces systematic ONT errors — particularly homopolymer indels — using a neural network trained on chemistry-specific signal patterns. The model must match the flow cell chemistry; one round of polishing is sufficient for high-depth data. For PacBio HiFi assemblies, polishing is generally unnecessary.

QUAST and BUSCO provide complementary assembly quality perspectives: contiguity and reference comparison (QUAST) versus gene-space completeness (BUSCO). Both should be run for any publication-quality assembly. Sniffles2 leverages the full read length to directly detect structural variants that short-read methods miss, and its efficient SNF format enables population-scale SV analysis through joint genotyping without re-running alignment.

The next notebook ([03: Isoform Analysis](./03_isoform_analysis.ipynb)) covers long-read transcriptomics with Oxford Nanopore direct RNA-seq and PacBio Iso-Seq, where read length similarly transforms what is measurable — enabling full-length isoform discovery, alternative splicing quantification, and direct detection of RNA modifications.

### Methods Summary

| Method | Tool | Use case | Key output | Notes |
|---|---|---|---|---|
| Repeat graph assembly | Flye | ONT or HiFi de novo | assembly.fasta + .gfa | Use --nano-hq for R10.4.1 |
| Haplotype-resolved assembly | Hifiasm | PacBio HiFi (± Hi-C / UL) | GFA → FASTA via awk | Trio binning for phased diploid |
| Neural-network polishing | Medaka | ONT assembly error correction | consensus.fasta | Match model to flow cell chemistry |
| Assembly evaluation | QUAST | Contiguity & reference comparison | N50, genome fraction, misassemblies | Use --reference if available |
| Gene completeness | BUSCO | Assembly gene-space check | C/D/F/M percentages | Select appropriate lineage database |
| Structural variant calling | Sniffles2 | SVs from aligned long reads | VCF with SVTYPE, SVLEN, AF | Use .snf for multi-sample joint calling |

In [ ]:
# ── Exercise 25-2: Assembly & Structural Variants ────────────────────────
#
# 1. ASSEMBLER CHOICE: You have 50x ONT R10.4.1 reads from a 12 Mb yeast genome.
#    Which Flye mode would you use? What genome size flag would you specify?
#    Under what circumstances would you prefer Hifiasm over Flye?
#
# 2. GFA -> FASTA conversion: The awk command to extract sequences from a GFA file is:
#       awk '/^S/{print ">"$2"\n"$3}' assembly.gfa > assembly.fasta
#    What does the /^S/ pattern match? What are fields $2 and $3 in a GFA S-line?
#
# 3. N50 CALCULATION (run the code below and answer the questions):

import numpy as np

def compute_n50(lengths):
    sorted_l = np.sort(np.array(lengths, dtype=int))[::-1]
    cumsum = np.cumsum(sorted_l)
    idx = np.searchsorted(cumsum, cumsum[-1] / 2)
    return int(sorted_l[idx])

def compute_ng50(lengths, genome_size):
    sorted_l = np.sort(np.array(lengths, dtype=int))[::-1]
    cumsum = np.cumsum(sorted_l)
    idx = np.searchsorted(cumsum, genome_size / 2)
    idx = min(idx, len(sorted_l) - 1)
    return int(sorted_l[idx])

# Assembly 1: highly fragmented
a1 = [500_000, 200_000, 150_000, 100_000, 80_000, 70_000, 50_000, 30_000, 20_000, 15_000]
# Assembly 2: two large contigs + small remainder
a2 = [5_500_000, 5_000_000, 300_000, 150_000, 50_000]

genome_size = 12_000_000  # 12 Mb yeast

print(f"Assembly 1 — N50: {compute_n50(a1):,} bp | NG50: {compute_ng50(a1, genome_size):,} bp")
print(f"Assembly 2 — N50: {compute_n50(a2):,} bp | NG50: {compute_ng50(a2, genome_size):,} bp")
# Q: Which assembly has higher N50? Which has higher NG50? Why do they differ?

# 4. SV FILTERING: Using the simulated SV data from Section 5 (df_sv),
#    write pandas expressions to:
#    (a) Keep only deletions (DEL) with SVLEN > 1000 and AF > 0.1
#    (b) Compute the median DEL size
#    (c) Count how many INS events fall in the Alu size range (250-320 bp)
#
# Uncomment and complete:
# dels_filtered = df_sv[___]
# median_del_size = ___
# alu_ins_count = ___

---

[← ONT Processing](./01_ont_processing.ipynb) | [Module Overview](../README.md) | [Next: Isoform Analysis →](./03_isoform_analysis.ipynb)

---

[← Previous: ONT Processing](./01_ont_processing.ipynb) | [Next: Isoform Analysis →](./03_isoform_analysis.ipynb)

---